Use `ast` to generate `DICT_DEFAULT`, `DICT_DESCRIPTIONS` and `DICT_MODULE` from `what_you_want.json`. Create each dict separately then combine into one and json.dump it.

In [1]:
import ast
import inspect
from dicts_test.data import example_module
from dicts_test.data.example_module import init_example_variables
from itertools import pairwise

In [2]:
# initialise dicts to use later
descriptions_dict = {}  # want to write the var name plus its comment to the descriptions dict
default_dict = {}
module_dict = {}

print(example_module.example_double)

init_example_variables()

print(example_module.example_double)
print(example_module.example_double_array_uninit)

None
0.0
None


In [3]:
x = ast.parse(inspect.getsource(example_module))

print(ast.dump(x, indent=4))

Module(
    body=[
        AnnAssign(
            target=Name(id='example_double', ctx=Store()),
            annotation=Name(id='float', ctx=Load()),
            value=Constant(value=None),
            simple=1),
        Expr(
            value=Constant(value='an example double')),
        AnnAssign(
            target=Name(id='example_double_nodescr', ctx=Store()),
            annotation=Name(id='float', ctx=Load()),
            value=Constant(value=None),
            simple=1),
        AnnAssign(
            target=Name(id='example_double_uninit', ctx=Store()),
            annotation=Name(id='float', ctx=Load()),
            value=Constant(value=None),
            simple=1),
        Expr(
            value=Constant(value='another example double (uninitialised)')),
        AnnAssign(
            target=Name(id='example_double_array', ctx=Store()),
            annotation=Subscript(
                value=Name(id='list', ctx=Load()),
                slice=Name(id='float', ctx=Load()),
  

In [4]:
# create DICT_MODULE part of dict (get all the variable names)
variable_names = []
print(f"initial var names list = {variable_names}")
for node in ast.walk(x):
    if isinstance(node, ast.AnnAssign):
        var_name = node.target.id
        if var_name not in variable_names:
            variable_names.append(var_name)
print(f"final var names list = {variable_names}")
print(f"number of variables = {len(variable_names)}")

dict_module_entry = {}
dict_module_entry["example_module"] = variable_names

print(f"dict_module_entry = {dict_module_entry}")
module_dict["DICT_MODULE"] = dict_module_entry
print(f"dict module = {module_dict}")

initial var names list = []
final var names list = ['example_double', 'example_double_nodescr', 'example_double_uninit', 'example_double_array', 'example_double_array_uninit', 'example_int', 'example_int_uninit', 'example_string', 'example_string_uninit']
number of variables = 9
dict_module_entry = {'example_module': ['example_double', 'example_double_nodescr', 'example_double_uninit', 'example_double_array', 'example_double_array_uninit', 'example_int', 'example_int_uninit', 'example_string', 'example_string_uninit']}
dict module = {'DICT_MODULE': {'example_module': ['example_double', 'example_double_nodescr', 'example_double_uninit', 'example_double_array', 'example_double_array_uninit', 'example_int', 'example_int_uninit', 'example_string', 'example_string_uninit']}}


In [5]:
# create DICT_DESCRIPTIONS part of dict (get the descriptions of the vars, if 
# no description then have description as "")
for node in ast.walk(x):
    if isinstance(node, ast.Expr):
        print(node.value.value)


an example double
another example double (uninitialised)
an example double array
and example integer
with a description over two lines

yet another integer
and example string
another example string (uninitialised)


In [6]:
from itertools import pairwise

# getting var descriptions - need to traverse, find AnnAssigns then find if there's an 
# Expr immediately after - if yes then this is the description, if no then put descprion
# as ""

var_names_and_descriptions = {}
for a, b in pairwise(x.body):
    if isinstance(a, ast.AnnAssign) and isinstance(b, ast.Expr):
        print(f"var name = {a.target.id} and description = {b.value.value}")
        var_names_and_descriptions[a.target.id] = b.value.value
    if isinstance(a, ast.AnnAssign) and not isinstance(b, ast.Expr):
        print(f"var_name = {a.target.id} and description = \"\"")
        var_names_and_descriptions[a.target.id] = ""

print(var_names_and_descriptions)
descriptions_dict["DICT_DESCRIPTIONS"] = var_names_and_descriptions
print(descriptions_dict)

# var_names_and_descriptions = {}
# for a, b in pairwise(x.body):
#     # need to sort ordering?
#     if isinstance(a, ast.AnnAssign):
#         if isinstance(b, ast.Expr):
#             print(f"var name = {a.target.id} and description = {b.value.value}")
#             var_names_and_descriptions[a.target.id] = b.value.value
#         else:
#             print(f"var_name = {a.target.id} and description = \"\"")
#             var_names_and_descriptions[a.target.id] = ""

# print(var_names_and_descriptions)
# descriptions_dict["DICT_DESCRIPTIONS"] = var_names_and_descriptions
# print(descriptions_dict)

var name = example_double and description = an example double
var_name = example_double_nodescr and description = ""
var name = example_double_uninit and description = another example double (uninitialised)
var name = example_double_array and description = an example double array
var_name = example_double_array_uninit and description = ""
var name = example_int and description = and example integer
with a description over two lines

var name = example_int_uninit and description = yet another integer
var name = example_string and description = and example string
var name = example_string_uninit and description = another example string (uninitialised)
{'example_double': 'an example double', 'example_double_nodescr': '', 'example_double_uninit': 'another example double (uninitialised)', 'example_double_array': 'an example double array', 'example_double_array_uninit': '', 'example_int': 'and example integer\nwith a description over two lines\n', 'example_int_uninit': 'yet another integer',

In [7]:
# create DICT_DEFAULT part of dict (get the initialised values of the vars)
initial_values_dict = {}
for node in ast.walk(x):
    if isinstance(node, ast.AnnAssign):
        print(node.value.value)
        # if node.value.value == None:
        #     dict_entry = "null"
        initial_values_dict[node.target.id] = node.value.value
    if isinstance(node, ast.Assign):
        print(node.value.value)
        initial_values_dict[node.targets[0].id] = node.value.value
print(f"initial_values_dict = {initial_values_dict}")
print(len(initial_values_dict))
default_dict["DICT_DEFAULT"] = initial_values_dict
print(f"dict default = {default_dict}")

None
None
None
None
None
None
None
None
None
0.0
15.0
5
string____
initial_values_dict = {'example_double': 0.0, 'example_double_nodescr': 15.0, 'example_double_uninit': None, 'example_double_array': None, 'example_double_array_uninit': None, 'example_int': 5, 'example_int_uninit': None, 'example_string': 'string____', 'example_string_uninit': None}
9
dict default = {'DICT_DEFAULT': {'example_double': 0.0, 'example_double_nodescr': 15.0, 'example_double_uninit': None, 'example_double_array': None, 'example_double_array_uninit': None, 'example_int': 5, 'example_int_uninit': None, 'example_string': 'string____', 'example_string_uninit': None}}


In [8]:
print(default_dict)
print(module_dict)
print(descriptions_dict)

{'DICT_DEFAULT': {'example_double': 0.0, 'example_double_nodescr': 15.0, 'example_double_uninit': None, 'example_double_array': None, 'example_double_array_uninit': None, 'example_int': 5, 'example_int_uninit': None, 'example_string': 'string____', 'example_string_uninit': None}}
{'DICT_MODULE': {'example_module': ['example_double', 'example_double_nodescr', 'example_double_uninit', 'example_double_array', 'example_double_array_uninit', 'example_int', 'example_int_uninit', 'example_string', 'example_string_uninit']}}
{'DICT_DESCRIPTIONS': {'example_double': 'an example double', 'example_double_nodescr': '', 'example_double_uninit': 'another example double (uninitialised)', 'example_double_array': 'an example double array', 'example_double_array_uninit': '', 'example_int': 'and example integer\nwith a description over two lines\n', 'example_int_uninit': 'yet another integer', 'example_string': 'and example string', 'example_string_uninit': 'another example string (uninitialised)'}}


In [9]:
import json
from pathlib import Path
new_dict = {**default_dict, **descriptions_dict, **module_dict}

with open(Path(Path.cwd(),"./data/trial_json.json").resolve(), "w") as f:
    json.dump(new_dict, f, indent=4)